# Evaluation: Wie gut verpixeln SAM 3, YuNet und SCRFD?

Wir vergleichen die **verpixelte Fläche** jeder Methode mit den **von Hand markierten Kopfmasken** (Ground Truth) aus X-AnyLabeling, über **alle markierten Frames**.

**Ablauf**
1. Frames exportieren: `uv run python scripts/export_gt_frames.py --frames 2500 3300 ...`
2. In X-AnyLabeling markieren (Ordner `data/interim/PAT_1/gt/`, Label `head`). **Jeder Frame mit `.json` wird ausgewertet.**
3. Pipeline-Masken erzeugen: `uv run python scripts/make_eval_masks.py` (nur fehlende; `--force` = alle neu)
4. Dieses Notebook von oben nach unten ausführen.

**Maße** (H = markierter Kopf, P = verpixelte Fläche):

| Maß | Formel | Frage | Ziel |
|---|---|---|---|
| Kopfabdeckung | \|P ∩ H\| / \|H\| | Wie viel vom Kopf ist verpixelt? | ≈ 100 % |
| Kern | wie oben, Kopf um `TOLERANZ_PX` verkleinert | dasselbe, ohne den unsicheren Rand (Kinn/Hals) | ≈ 100 % |
| geschützt | Kopfabdeckung ≥ `GESCHUETZT_AB` | Ist der Kopf (praktisch) ganz verpixelt? | alle |
| Frames geschützt | alle Köpfe des Frames geschützt | Ist der ganze Frame sicher? | alle |
| Überverpixelung | \|P \ H\| / \|H\| | Wie viel zusätzliche Fläche, gemessen an der Kopfgröße? | klein |
| Präzision | \|P ∩ H\| / \|P\| | Welcher Anteil der verpixelten Fläche ist wirklich Kopf? | groß |

Alles **getrennt nach Farbe (F) und S/W (SW)** und für beide zusammen.

**Zwei Ebenen**
- **Pipeline** (Hauptergebnis): die Fläche, die das Verfahren im fertigen Video wirklich verpixelt, inklusive hold/trail/Tracking/Lücken füllen. Kommt aus `data/interim/PAT_1/eval/masks/` (Schritt 3 oben).
- **Detektor** (optional, `MIT_DETEKTOR`): nur die Erkennung in diesem einen Bild, live berechnet. Zeigt, wie viel die Nachbearbeitung rettet.

**Datenschutz:** Das Notebook zeigt keine Bilder an. Kontrollbilder werden nur als Datei in `data/processed/PAT_1/eval/` gespeichert. Vor jedem Commit die Ausgaben löschen.

## Schritt 1: Setup, Pfade und Einstellungen

- `VIEWS`: die beiden Ansichten wie in allen Verfahrens-Notebooks. Ausgewertet wird nur **innerhalb** dieser Rechtecke.
- `GESCHUETZT_AB`: ab welcher Abdeckung ein Kopf als geschützt gilt.
- `TOLERANZ_PX`: um so viele Pixel wird die Kopfmaske für die „Kern“-Abdeckung verkleinert.
- `MIT_DETEKTOR`: `True` = zusätzlich die Detektor-Ebene rechnen (SAM 3 braucht dann die GPU, Laden ca. 20 s, danach ca. 1 s pro Frame). `False` = nur die Pipelines (schnell).
- `KONTROLLBILDER`: `True` = pro Frame und Methode ein Kontrollbild speichern.

In [ ]:
import csv
import json
from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = Path.cwd().parent                     # Notebook liegt in notebooks/
VIDEO_STEM = "VID_76"
GT_DIR   = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "gt"           # Bilder + JSON aus X-AnyLabeling
MASK_DIR = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "eval" / "masks"  # Pipeline-Masken (make_eval_masks.py)
EVAL_DIR = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "eval"         # Ergebnis-Tabellen (CSV)
VIS_DIR  = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "eval"       # Kontrollbilder (nur lokal ansehen)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

H, W = 1080, 1920
VIEWS = {"F": (0, 0, 961, 1080), "SW": (991, 10, 1910, 527)}   # wie in den Verfahrens-Notebooks
VIEW_NAMES = {"F": "Farbe", "SW": "S/W", "beide": "beide"}

GESCHUETZT_AB  = 0.95    # Kopf gilt als geschützt ab 95 % Abdeckung
TOLERANZ_PX    = 5       # Kopfmaske für die "Kern"-Abdeckung um 5 px verkleinern
MIT_DETEKTOR   = True    # zusätzlich Detektor-Ebene live rechnen (langsamer, SAM 3 braucht GPU)
KONTROLLBILDER = True    # Kontrollbilder speichern

GT_FILES = sorted(GT_DIR.glob(f"{VIDEO_STEM}_f*.json"))
FRAMES = [int(p.stem.split("_f")[-1]) for p in GT_FILES]
print(f"{len(FRAMES)} markierte Frames: {FRAMES}")
for f in FRAMES:
    if not (GT_DIR / f"{VIDEO_STEM}_f{f:05d}.png").exists():
        print(f"WARNUNG: Bild zu Frame {f} fehlt")

VIEW_MASK = {}
for v, (x0, y0, x1, y1) in VIEWS.items():
    VIEW_MASK[v] = np.zeros((H, W), dtype=bool)
    VIEW_MASK[v][y0:y1, x0:x1] = True
ANALYSED = VIEW_MASK["F"] | VIEW_MASK["SW"]   # alles, was ausgewertet wird

## Schritt 2: Ground Truth laden

Aus jeder JSON-Datei wird pro Polygon mit Label `head` eine Pixelmaske gezeichnet (`cv2.fillPoly`).

- **Überlappende Polygone werden zu einem Kopf vereint.** So kannst du fehlende Teile als zusätzliches Polygon ergänzen.
- **Ansicht:** Jeder Kopf wird über seinen Mittelpunkt der Ansicht F oder SW zugeordnet und auf diese Ansicht zugeschnitten.
- Ein Frame **ohne** Kopf (leere Markierung) zählt trotzdem: Dort ist jede Verpixelung Überverpixelung.

In [ ]:
def load_gt(json_path, label="head"):
    # Liest die Polygone aus X-AnyLabeling und gibt eine Liste von Köpfen zurück
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    polys = []
    for s in data["shapes"]:
        if s["label"] != label or s["shape_type"] != "polygon":
            continue
        m = np.zeros((H, W), dtype=np.uint8)
        cv2.fillPoly(m, [np.round(np.array(s["points"])).astype(np.int32)], 1)
        polys.append(m.astype(bool))

    groups = []                                   # überlappende Polygone zusammenfassen
    for m in polys:
        hits = [g for g in groups if (g & m).any()]
        for g in hits:
            m = m | g
        groups = [g for g in groups if not any(g is h for h in hits)] + [m]

    heads = []
    for m in groups:
        ys, xs = np.nonzero(m)
        cx, cy = xs.mean(), ys.mean()
        view = next((v for v, (x0, y0, x1, y1) in VIEWS.items() if x0 <= cx < x1 and y0 <= cy < y1), None)
        if view is None:
            print(f"   WARNUNG: Kopf bei x={cx:.0f}, y={cy:.0f} liegt in keiner Ansicht -> ignoriert")
            continue
        m = m & VIEW_MASK[view]
        ys, xs = np.nonzero(m)
        heads.append({"view": view, "mask": m, "area": int(m.sum()),
                      "box": [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]})
    heads.sort(key=lambda h: (h["view"], h["box"][0]))
    for k, h in enumerate(heads):
        h["nr"] = k
    return heads, len(polys)


GT = {}
for f, p in zip(FRAMES, GT_FILES):
    GT[f], n_polys = load_gt(p)
    desc = ", ".join(f"{VIEW_NAMES[h['view']]} {h['area']} px" for h in GT[f]) or "keine"
    print(f"Frame {f}: {n_polys} Polygone -> {len(GT[f])} Köpfe ({desc})")
n_heads = {v: sum(h["view"] == v for f in FRAMES for h in GT[f]) for v in VIEWS}
print(f"\nKöpfe gesamt: Farbe {n_heads['F']}, S/W {n_heads['SW']}")

## Schritt 3: Detektoren (nur für die Detektor-Ebene)

Gleiche Einstellungen wie in den Verfahrens-Notebooks, aber **ohne** Zeitlogik (nur dieses eine Bild):

| Methode | Erkennung | verpixelte Fläche |
|---|---|---|
| **YuNet** (`YuNet.ipynb`, Schritt 7) | pro Ansicht, Score ≥ 0,3 | Gesichtsbox + 50 % Rand links/rechts/unten, 80 % oben |
| **SCRFD-10G** (`SCRFD-10G.ipynb`, Abschnitt 16) | pro Ansicht (Farbe 640 × 736, S/W 928 × 544), Score ≥ 0,5 | Gesichtsbox + 50 % Rand an allen Seiten |
| **SAM 3** (`SAM3.ipynb`, `run_video_detect`) | pro Ansicht, Prompt „head“, Score ≥ 0,5 | Maske + 8 px; zu große Maske (> 60 % der Ansicht) unverändert |

Bei `MIT_DETEKTOR = False` können diese drei Zellen übersprungen werden.

In [ ]:
# ---------- YuNet ----------
YUNET_MODEL = PROJECT_ROOT / "models" / "face_detection_yunet_2023mar.onnx"
YUNET = {"score_threshold": 0.3, "padding": 0.5, "padding_top": 0.8}


def pixel_area_yunet(img, cfg=YUNET):
    # Jede Ansicht einzeln, Rand an der Ansicht abgeschnitten (wie pixelate_region)
    P, dets = np.zeros((H, W), dtype=bool), []
    for v, (x0, y0, x1, y1) in VIEWS.items():
        vw, vh = x1 - x0, y1 - y0
        det = cv2.FaceDetectorYN.create(str(YUNET_MODEL), "", (vw, vh), score_threshold=cfg["score_threshold"],
                                        nms_threshold=0.3, top_k=5000)
        _, faces = det.detect(np.ascontiguousarray(img[y0:y1, x0:x1]))
        for f in ([] if faces is None else faces):
            x, y, w, h = f[:4].astype(int)
            rx1, ry1 = max(x - int(w * cfg["padding"]), 0), max(y - int(h * cfg["padding_top"]), 0)
            rx2, ry2 = min(x + w + int(w * cfg["padding"]), vw), min(y + h + int(h * cfg["padding"]), vh)
            P[y0 + ry1:y0 + ry2, x0 + rx1:x0 + rx2] = True
            dets.append({"view": v, "score": float(f[-1]), "box": [int(x) + x0, int(y) + y0, int(x + w) + x0, int(y + h) + y0]})
    return P, dets

In [ ]:
# ---------- SCRFD-10G ----------
import onnxruntime as ort

ort.set_default_logger_severity(3)   # harmlose Warnungen zu Ausgabegrößen ausblenden
SCRFD_MODEL = PROJECT_ROOT / "models" / "buffalo_l" / "buffalo_l" / "det_10g.onnx"
SCRFD_CFG = {"score_threshold": 0.5, "padding": 0.5}
SCRFD_SIZES = {"F": (640, 736), "SW": (928, 544)}


class SCRFD:
    # Gekürzte Kopie aus SCRFD-10G.ipynb (Letterbox, 3 Maßstäbe, NMS)
    def __init__(self, model_path, input_size=(1280, 736), score_threshold=0.5, nms_threshold=0.4, pad_value=0):
        self.in_w, self.in_h = input_size
        self.score_threshold, self.nms_threshold, self.pad_value = score_threshold, nms_threshold, pad_value
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]
        scale = min(self.in_w / iw, self.in_h / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        canvas = np.full((self.in_h, self.in_w, 3), self.pad_value, dtype=np.uint8)
        canvas[:new_h, :new_w] = cv2.resize(img_bgr, (new_w, new_h))
        blob = cv2.dnn.blobFromImage(canvas, 1.0 / 128, (self.in_w, self.in_h), (127.5, 127.5, 127.5), swapRB=True)
        outs = self.session.run(self.output_names, {self.input_name: blob})
        boxes, scores = [], []
        for idx, stride in enumerate([8, 16, 32]):
            s = outs[idx].reshape(-1)
            d = outs[idx + 3].reshape(-1, 4) * stride
            gh, gw = self.in_h // stride, self.in_w // stride
            ys, xs = np.mgrid[:gh, :gw]
            centers = np.repeat(np.stack([xs, ys], axis=-1).astype(np.float32).reshape(-1, 2) * stride, 2, axis=0)
            for k in np.where(s >= self.score_threshold)[0]:
                cx, cy = centers[k]
                x1, y1, x2, y2 = cx - d[k, 0], cy - d[k, 1], cx + d[k, 2], cy + d[k, 3]
                boxes.append([x1 / scale, y1 / scale, (x2 - x1) / scale, (y2 - y1) / scale])
                scores.append(float(s[k]))
        if not boxes:
            return np.empty((0, 5), dtype=np.float32)
        idxs = np.array(cv2.dnn.NMSBoxes(boxes, scores, self.score_threshold, self.nms_threshold)).reshape(-1)
        return np.array([[*boxes[i], scores[i]] for i in idxs], dtype=np.float32)


_scrfd = {}


def pixel_area_scrfd(img, cfg=SCRFD_CFG):
    # Jede Ansicht einzeln, Rand an der Ansicht abgeschnitten (wie Abschnitt 16 in SCRFD-10G.ipynb)
    P, dets = np.zeros((H, W), dtype=bool), []
    p = cfg["padding"]
    for v, (vx0, vy0, vx1, vy1) in VIEWS.items():
        if v not in _scrfd:
            _scrfd[v] = SCRFD(SCRFD_MODEL, input_size=SCRFD_SIZES[v], score_threshold=cfg["score_threshold"])
        for x, y, w, h, s in _scrfd[v].detect(np.ascontiguousarray(img[vy0:vy1, vx0:vx1])):
            x, y = x + vx0, y + vy0
            x0, x1 = int(max(vx0, x - p * w)), int(min(vx1, x + w + p * w))
            y0, y1 = int(max(vy0, y - p * h)), int(min(vy1, y + h + p * h))
            P[y0:y1, x0:x1] = True
            dets.append({"view": v, "score": float(s), "box": [int(x), int(y), int(x + w), int(y + h)]})
    return P, dets


print("SCRFD-Modell:", "gefunden" if SCRFD_MODEL.exists() else f"FEHLT ({SCRFD_MODEL})")

In [ ]:
# ---------- SAM 3 (braucht GPU und: uv sync --group sam3) ----------
import torch
from transformers import Sam3Model, Sam3Processor

SAM3_DIR = PROJECT_ROOT / "models" / "sam3"
SAM3_CFG = {"prompt": "head", "det_thr": 0.5, "dilate": 8, "max_frac": 0.6}   # wie run_video_detect
_sam3 = {}


def pixel_area_sam3(img, cfg=SAM3_CFG):
    if not _sam3:   # Modell nur einmal laden
        _sam3["proc"] = Sam3Processor.from_pretrained(SAM3_DIR)
        _sam3["model"] = Sam3Model.from_pretrained(SAM3_DIR, dtype=torch.bfloat16).to("cuda").eval()
    processor, model = _sam3["proc"], _sam3["model"]
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * cfg["dilate"] + 1, 2 * cfg["dilate"] + 1))
    P, dets = np.zeros((H, W), dtype=bool), []
    for v, (x0, y0, x1, y1) in VIEWS.items():
        vw, vh = x1 - x0, y1 - y0
        inputs = processor(images=cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2RGB), text=cfg["prompt"],
                           return_tensors="pt").to("cuda", dtype=torch.bfloat16)
        with torch.no_grad():
            out = model(**inputs)
        r = processor.post_process_instance_segmentation(
            out, threshold=cfg["det_thr"], mask_threshold=0.5, target_sizes=inputs["original_sizes"].tolist())[0]
        for m, b, s in zip(r["masks"].cpu().numpy().astype(bool), r["boxes"].cpu().tolist(),
                           r["scores"].float().cpu().tolist()):
            big = (b[2] - b[0]) > cfg["max_frac"] * vw or (b[3] - b[1]) > cfg["max_frac"] * vh
            if not big:
                m = cv2.dilate(m.astype(np.uint8), kernel).astype(bool)
            P[y0:y1, x0:x1] |= m
            dets.append({"view": v, "score": s, "box": [int(b[0]) + x0, int(b[1]) + y0, int(b[2]) + x0, int(b[3]) + y0]})
    return P, dets


def sam3_free():
    _sam3.clear()                  # Modell vergessen -> GPU-Speicher wieder frei
    torch.cuda.empty_cache()

## Schritt 4: Verpixelte Flächen sammeln

Für jeden markierten Frame:
1. **Pipeline**: Masken aus `data/interim/PAT_1/eval/masks/` laden. Fehlt eine, wird sie gemeldet → `uv run python scripts/make_eval_masks.py` ausführen.
2. **Detektor** (nur bei `MIT_DETEKTOR = True`): die Detektoren live auf dem Bild rechnen.

Ausgegeben werden nur Zahlen (verpixelte Pixel pro Methode).

In [ ]:
PIPELINES = {"SAM 3 Pipeline": "sam3det", "YuNet Pipeline": "yunet", "SCRFD Pipeline": "scrfd"}
DETECTORS = {}
if MIT_DETEKTOR:
    DETECTORS = {"SAM 3 Detektor": pixel_area_sam3, "YuNet Detektor": pixel_area_yunet}
    if SCRFD_MODEL.exists():
        DETECTORS["SCRFD Detektor"] = pixel_area_scrfd

areas = {f: {} for f in FRAMES}   # areas[Frame][Methode] = Maske (True = verpixelt)
missing = {}
for f in FRAMES:
    for name, key in PIPELINES.items():
        mf = MASK_DIR / f"{VIDEO_STEM}_f{f:05d}_{key}.npz"
        if mf.exists():
            areas[f][name] = np.load(mf)["mask"] & ANALYSED      # nur innerhalb der Ansichten zählt
        else:
            missing.setdefault(name, []).append(f)
    if DETECTORS:
        img = cv2.imread(str(GT_DIR / f"{VIDEO_STEM}_f{f:05d}.png"))   # wird NICHT angezeigt
        for name, fn in DETECTORS.items():
            P, _ = fn(img)
            areas[f][name] = P & ANALYSED
    print(f"Frame {f}: " + " | ".join(f"{n} {int(P.sum())} px" for n, P in areas[f].items()))
if "SAM 3 Detektor" in DETECTORS:
    sam3_free()

for name, fs in missing.items():
    print(f"ACHTUNG: {name} fehlt für Frames {fs} -> scripts/make_eval_masks.py ausführen")
METHODS = [m for m in [*PIPELINES, *DETECTORS] if any(m in areas[f] for f in FRAMES)]
print("\nAusgewertete Methoden:", METHODS)

## Schritt 5: Maße berechnen

**Pro Kopf:** Kopfabdeckung (voll und Kern), geschützt ja/nein.

**Gesamt** pro Methode und Ansicht (über alle Frames):
- `Abdeckung`: alle getroffenen Kopfpixel / alle Kopfpixel (große Köpfe zählen mehr)
- `Median` / `Min`: Abdeckung des mittleren bzw. schlechtesten **Kopfes** (jeder Kopf zählt gleich)
- `Kern`: mittlere Kern-Abdeckung pro Kopf
- `geschützt`: geschützte Köpfe / alle Köpfe
- `Frames ok`: Frames, in denen **alle** Köpfe der Ansicht geschützt sind / Frames mit Kopf
- `Überverp.`: verpixelte Fläche außerhalb der Köpfe / Kopffläche (0,5 = zusätzlich die halbe Kopffläche)
- `Präzision`: Anteil der verpixelten Fläche, der wirklich Kopf ist

Danach eine Liste aller **nicht geschützten Köpfe**, also die Stellen, die man sich im Kontrollbild ansehen sollte.

In [ ]:
CORE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * TOLERANZ_PX + 1, 2 * TOLERANZ_PX + 1))
for f in FRAMES:
    for h in GT[f]:
        core = cv2.erode(h["mask"].astype(np.uint8), CORE_KERNEL).astype(bool)
        h["core"] = core if core.any() else h["mask"]      # sehr kleiner Kopf: volle Maske nehmen

head_rows, view_rows = [], []
for f in FRAMES:
    for name, P in areas[f].items():
        hr = []
        for h in GT[f]:
            cov = (P & h["mask"]).sum() / h["area"]
            hr.append({"frame": f, "methode": name, "kopf": h["nr"], "ansicht": h["view"], "flaeche_px": h["area"],
                       "abdeckung": float(cov), "abdeckung_kern": float((P & h["core"]).sum() / h["core"].sum()),
                       "geschuetzt": bool(cov >= GESCHUETZT_AB)})
        head_rows += hr
        for v in VIEWS:
            Hm = np.zeros((H, W), dtype=bool)
            for h in GT[f]:
                if h["view"] == v:
                    Hm |= h["mask"]
            Pv = P & VIEW_MASK[v]
            view_rows.append({"frame": f, "methode": name, "ansicht": v,
                              "koepfe": sum(h["view"] == v for h in GT[f]), "kopf_px": int(Hm.sum()),
                              "treffer_px": int((Pv & Hm).sum()), "ueber_px": int((Pv & ~Hm).sum()),
                              "alle_geschuetzt": all(r["geschuetzt"] for r in hr if r["ansicht"] == v)})


def summarize(name, v):
    vr = [r for r in view_rows if r["methode"] == name and (v == "beide" or r["ansicht"] == v)]
    hr = [r for r in head_rows if r["methode"] == name and (v == "beide" or r["ansicht"] == v)]
    kopf, treffer, ueber = (sum(r[k] for r in vr) for k in ["kopf_px", "treffer_px", "ueber_px"])
    per_frame = {}
    for r in vr:
        if r["koepfe"]:
            per_frame.setdefault(r["frame"], []).append(r["alle_geschuetzt"])
    covs = [r["abdeckung"] for r in hr]
    nan = float("nan")
    return {"methode": name, "ansicht": v, "frames": len({r["frame"] for r in vr}), "koepfe": len(hr),
            "abdeckung": treffer / kopf if kopf else nan,
            "abdeckung_median": float(np.median(covs)) if covs else nan,
            "abdeckung_min": float(np.min(covs)) if covs else nan,
            "kern_mittel": float(np.mean([r["abdeckung_kern"] for r in hr])) if hr else nan,
            "geschuetzt": sum(r["geschuetzt"] for r in hr),
            "frames_ok": sum(all(x) for x in per_frame.values()), "frames_mit_kopf": len(per_frame),
            "ueberverpixelung": ueber / kopf if kopf else nan,
            "praezision": treffer / (treffer + ueber) if treffer + ueber else nan}


gesamt = [summarize(m, v) for m in METHODS for v in ["F", "SW", "beide"]]

print(f"{'Methode':15s} {'Ansicht':7s} {'Köpfe':>5s} {'Abdeckung':>9s} {'Median':>7s} {'Min':>6s} {'Kern':>6s} "
      f"{'geschützt':>9s} {'Frames ok':>9s} {'Überverp.':>9s} {'Präzision':>9s}")
for r in gesamt:
    print(f"{r['methode']:15s} {VIEW_NAMES[r['ansicht']]:7s} {r['koepfe']:5d} {r['abdeckung']:9.1%} "
          f"{r['abdeckung_median']:7.1%} {r['abdeckung_min']:6.1%} {r['kern_mittel']:6.1%} "
          f"{r['geschuetzt']:4d}/{r['koepfe']:<4d} {r['frames_ok']:4d}/{r['frames_mit_kopf']:<4d} "
          f"{r['ueberverpixelung']:9.2f} {r['praezision']:9.1%}")
    if r["ansicht"] == "beide":
        print()

print("Nicht geschützte Köpfe (Abdeckung < {:.0%}):".format(GESCHUETZT_AB))
bad = [r for r in head_rows if not r["geschuetzt"]]
for r in sorted(bad, key=lambda r: (r["methode"], r["frame"])):
    print(f"   {r['methode']:15s} Frame {r['frame']:5d}  Kopf {r['kopf']} ({VIEW_NAMES[r['ansicht']]}, {r['flaeche_px']} px)  "
          f"Abdeckung {r['abdeckung']:.1%}, Kern {r['abdeckung_kern']:.1%}")
if not bad:
    print("   keine")

## Schritt 6: Speichern

- **CSV-Tabellen** in `data/interim/PAT_1/eval/` (Semikolon, Dezimalkomma → direkt in Excel öffnen):
  - `pro_kopf.csv`: jeder Kopf in jedem Frame für jede Methode
  - `pro_frame_ansicht.csv`: Pixelzahlen pro Frame, Ansicht und Methode
  - `gesamt.csv`: die Gesamttabelle aus Schritt 5
- **Kontrollbilder** in `data/processed/PAT_1/eval/f<Frame>_<methode>.png`: grün = markierter Kopf, rot = verpixelt. Nur **lokal** ansehen, enthält Patientendaten.

In [ ]:
def save_csv(rows, path):
    def fmt(x):
        return f"{x:.4f}".replace(".", ",") if isinstance(x, float) else x   # Dezimalkomma für Excel
    with open(path, "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()), delimiter=";")
        w.writeheader()
        w.writerows({k: fmt(v) for k, v in r.items()} for r in rows)
    print("gespeichert:", path.relative_to(PROJECT_ROOT))


save_csv(head_rows, EVAL_DIR / "pro_kopf.csv")
save_csv(view_rows, EVAL_DIR / "pro_frame_ansicht.csv")
save_csv(gesamt, EVAL_DIR / "gesamt.csv")

if KONTROLLBILDER:
    for f in FRAMES:
        frame = cv2.imread(str(GT_DIR / f"{VIDEO_STEM}_f{f:05d}.png"))
        for name, P in areas[f].items():
            vis = frame.copy()
            vis[~ANALYSED] = (vis[~ANALYSED] * 0.25).astype(np.uint8)                 # nicht ausgewertet: dunkel
            vis[P] = (0.5 * vis[P] + 0.5 * np.array([0, 0, 255])).astype(np.uint8)    # rot: verpixelt
            for h in GT[f]:
                cnts, _ = cv2.findContours(h["mask"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                cv2.drawContours(vis, cnts, -1, (0, 255, 0), 2)                        # grün: Ground Truth
            cv2.imwrite(str(VIS_DIR / f"f{f:05d}_{name.replace(' ', '').lower()}.png"), vis)
    print(f"Kontrollbilder: {len(FRAMES)} Frames x {len(METHODS)} Methoden in", VIS_DIR.relative_to(PROJECT_ROOT))